In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from load import (
    colunas_renomeadas,
    map_estados,
    map_regiao,
    cols_sono,
    cols_sintomas_psicologicos,  
    cols_drogas,
    cols_atividade,
    cols_dieta,
    cols_deficiencia,
    cols_gravidez,
    cols_comorbidades,           
    cols_violencia,
    cols_servicos_saude,
    cols_trabalho_renda_demografia,
    cols_antropometria,
    carregar_colunas
)

In [32]:
df = pd.read_csv('../data/pns2019/pns2019.csv', low_memory=False)
print(f"Shape bruto: {df.shape}")

Shape bruto: (293726, 1087)


In [45]:
# Renomeia colunas para nomes legíveis
df.rename(columns=colunas_renomeadas, inplace=True)

# Mapeia código UF → nome do estado e estado → grande região
df["Estado"] = df["Estado"].astype(str).map(map_estados)
df["Regiao"] = df["Estado"].map(map_regiao)

# Filtra apenas entrevistas realizadas (V0015 == 1)
df_respondente = df[df["V0015"] == 1]

# Restringe à faixa etária do estudo: 30 a 59 anos
df_faixa_etaria = df_respondente[
    (df_respondente["Idade"] >= 30) & (df_respondente["Idade"] < 60)
]

print(f"Respondentes totais:        {len(df_respondente):,}")
print(f"Faixa 30–59 anos:           {len(df_faixa_etaria):,}")

Respondentes totais:        279,382
Faixa 30–59 anos:           114,281


In [43]:

# Lista Única Consolidada
val_sel = carregar_colunas()



# Remove duplicatas mantendo a ordem
val_sel = list(dict.fromkeys(val_sel))

# Verifica quais colunas existem no DataFrame (segurança contra colunas ausentes)
cols_existentes     = [c for c in val_sel if c in df_faixa_etaria.columns]
cols_inexistentes   = [c for c in val_sel if c not in df_faixa_etaria.columns]

print(f"Variáveis solicitadas:   {len(val_sel)}")
print(f"Variáveis encontradas:   {len(cols_existentes)}")
if cols_inexistentes:
    print(f"Variáveis NÃO encontradas ({len(cols_inexistentes)}): {cols_inexistentes}")

df_aplicacao = df_faixa_etaria[cols_existentes].copy()
print(f"\nShape df_aplicacao: {df_aplicacao.shape}")

Variáveis solicitadas:   131
Variáveis encontradas:   130
Variáveis NÃO encontradas (1): ['D009a']

Shape df_aplicacao: (114281, 130)


In [58]:

cols = ["Q00201","Q03001","Q060","Q06306","Q068",
        "Q074","Q079","Q088","Q092","Q11006","Q11604","Q120","Q128","Q124","Q084"]

df_toc = df_aplicacao[df_aplicacao["TOC"] == 1].copy()

# Substitui NaN por 2 (assume "Não" para ausência de resposta)
df_temp = df_aplicacao.copy()

# Filtra: todas comorbidades == 2 AND todas doenças == 2
mask_sem_doenca = (
    (df_temp[cols] == 2).all(axis=1) 
)

df_sem_doenca = df_aplicacao[mask_sem_doenca].copy()

print(f"Grupo TOC (casos):            {len(df_toc):,} registros")
print(f"Grupo saudável (controle):    {len(df_sem_doenca):,} registros")
print(f"Total:                         {len(df_toc) + len(df_sem_doenca):,} registros")

Grupo TOC (casos):            209 registros
Grupo saudável (controle):    19,664 registros
Total:                         19,873 registros


In [59]:
df_juncao = pd.concat([df_toc, df_sem_doenca], ignore_index=True)
df_juncao.drop(columns=cols, inplace=True)

# Remove índice residual se existir
if 'Unnamed: 0' in df_juncao.columns:
    df_juncao.drop(columns=['Unnamed: 0'], inplace=True)

# Garante que o grupo controle (TOC == NaN) seja marcado como TOC == 2 (sem TOC)
df_juncao.loc[df_juncao['TOC'].isna(), 'TOC'] = 2

print(f"Shape df_juncao: {df_juncao.shape}")
print(f"TOC == 1 (casos):     {(df_juncao['TOC'] == 1).sum():,}")
print(f"TOC == 2 (controle):  {(df_juncao['TOC'] == 2).sum():,}")
df_juncao.head(3)


Shape df_juncao: (19873, 115)
TOC == 1 (casos):     209
TOC == 2 (controle):  19,664


,N010,N011,N012,N013,N014,N015,N016,N017,N018,P027,...,E01602,VDF00102,M005011,M00601,M007,M011071,Peso,Altura,W00103,W00203
0,4.0,4.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,60.0,155.0,NaN,NaN
1,3.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,1.0,...,1500.0,NaN,NaN,NaN,NaN,2.0,61.0,157.0,NaN,NaN
2,4.0,1.0,1.0,1.0,3.0,1.0,2.0,3.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,73.0,158.0,NaN,NaN


In [50]:
todas_colunas = df_juncao.columns.tolist()

# ── Quantitativas Contínuas
quant_continuas = [
    c for c in todas_colunas
    if c.startswith(('VDF', 'E016', 'E018', 'W0', 'Peso', 'Altura'))
]

# ── Quantitativas Discretas
_discretas_fixas = [
    'Idade',      # Idade em anos completos
    'P035',       # Dias/semana de exercício
    'E019',       # Horas trabalhadas por semana
    'P029',       # Doses de álcool por dia
    'P03701',     # Horas de exercício por dia
    'P03702',     # Minutos de exercício por dia
    'P03904',     # Dias/semana de atividade física no trabalho
    'P05801',     # Cigarros/semana (ex-fumantes)
    'M005011',    # Horas em turno noturno por dia
]
quant_discretas = [c for c in _discretas_fixas if c in todas_colunas]

# ── Qualitativas Ordinais
_ordinais_fixas = [
    'N010',       # Qualidade do sono
    'N011', 'N012', 'N013', 'N014',  # PHQ-9 adaptado
    'N015', 'N016', 'N017', 'N018',
    'J001',       # Avaliação geral de saúde
    'J00101',     # Avaliação bem-estar físico/mental
    'J01101',     # Última consulta médica
    'M00601',     # Frequência de trabalho noturno
    'D009a',      # Nível de escolaridade
    'P019',       # Vezes/dia de frutas
    'P01001',     # Vezes/dia de verduras
]
# Prefixos que indicam ordinal
_ordinais_por_prefixo = [
    c for c in todas_colunas
    if c.startswith('G')                          # Deficiências (escala 1–4)
    or (c.startswith('P006') and c != 'P06701')   # Dias/semana alimentos
    or c.startswith('P028')                        # Faixas de álcool
    or c in ('P02501', 'P02601', 'P02602',         # Frequência doces, sal, lanche
              'P027', 'P02801',                    # Frequência álcool
              'P01101', 'P01601', 'P02001',        # Frequência carnes, sucos, refrig.
              'P02002', 'P013', 'P015', 'P018',    # Frequência frango, peixe, frutas
              'P00901', 'P023')
]
qual_ordinais = list(dict.fromkeys(
    [c for c in _ordinais_fixas if c in todas_colunas] + _ordinais_por_prefixo
))
qual_ordinais = [c for c in qual_ordinais if c in todas_colunas]

# ── Qualitativas Nominais: tudo o que sobrar
usadas = set(quant_continuas + quant_discretas + qual_ordinais)
qual_nominais = [c for c in todas_colunas if c not in usadas]

# ── Agrupamentos finais
quantitativas = quant_continuas + quant_discretas
qualitativas  = qual_ordinais   + qual_nominais

# Garante tipo numérico nas quantitativas
for c in quantitativas:
    df_juncao[c] = pd.to_numeric(df_juncao[c], errors='coerce')

print("─" * 50)
print(f"Total de variáveis mapeadas:     {len(todas_colunas)}")
print(f"  Quantitativas Contínuas:       {len(quant_continuas)}")
print(f"  Quantitativas Discretas:       {len(quant_discretas)}")
print(f"  Qualitativas Ordinais:         {len(qual_ordinais)}")
print(f"  Qualitativas Nominais:         {len(qual_nominais)}")
print("─" * 50)

──────────────────────────────────────────────────
Total de variáveis mapeadas:     130
  Quantitativas Contínuas:       6
  Quantitativas Discretas:       9
  Qualitativas Ordinais:         63
  Qualitativas Nominais:         52
──────────────────────────────────────────────────


In [51]:
def q1(x):        return x.quantile(0.25)
def q3(x):        return x.quantile(0.75)
def vazios(x):    return int(x.isnull().sum())
def nao_nulos(x): return int(x.notna().sum())

if quantitativas:
    tabela_quant = (
        df_juncao[quantitativas]
        .agg(['min', 'max', 'mean', 'median', 'std', 'var', vazios, nao_nulos, q1, q3])
        .T
        .rename(columns={
            'min':      'Mínimo',
            'max':      'Máximo',
            'mean':     'Média',
            'median':   'Mediana',
            'std':      'Desvio Padrão',
            'var':      'Variância',
            'vazios':   'Qtd Vazios',
            'nao_nulos':'N Válidos',
            'q1':       'Q1 (25%)',
            'q3':       'Q3 (75%)',
        })
    )
    # Adiciona coluna de subtipo
    tabela_quant.insert(0, 'Subtipo', [
        'Contínua' if c in quant_continuas else 'Discreta'
        for c in tabela_quant.index
    ])
    tabela_quant.index.name = 'Variável'
    tabela_quant = tabela_quant.round(2)
else:
    tabela_quant = pd.DataFrame()

print(f"Variáveis quantitativas: {len(tabela_quant)}")
display(tabela_quant)

Variáveis quantitativas: 15


Variáveis quantitativas: 15


,Subtipo,Mínimo,Máximo,Média,Mediana,Desvio Padrão,Variância,Qtd Vazios,N Válidos,Q1 (25%),Q3 (75%)
Variável,,,,,,,,,,,
E01602,Contínua,4.0,100000.0,2384.44,1500.0,3334.59,11119481.00,4369.0,15504.0,998.0,2500.00
VDF00102,Contínua,1.0,80000.0,470.66,180.0,2559.87,6552910.17,17031.0,2842.0,120.0,350.00
Peso,Contínua,32.0,160.0,73.28,72.0,14.58,212.57,186.0,19687.0,63.0,82.00
Altura,Contínua,130.0,202.0,166.43,166.0,9.57,91.61,186.0,19687.0,160.0,173.00
W00103,Contínua,34.9,134.9,74.42,72.9,14.76,217.96,18477.0,1396.0,64.0,83.72
W00203,Contínua,135.0,194.8,166.42,166.8,9.32,86.89,18477.0,1396.0,160.0,173.20
Idade,Discreta,30.0,59.0,41.81,40.0,8.20,67.17,0.0,19873.0,35.0,48.00
P035,Discreta,0.0,7.0,3.29,3.0,1.88,3.52,11291.0,8582.0,2.0,5.00
E019,Discreta,1.0,80.0,18.47,18.0,12.14,147.44,19014.0,859.0,9.0,24.00


In [52]:
registros_quali = []

for col in qualitativas:
    serie      = df_juncao[col]
    n_total    = len(serie)
    n_nulos    = int(serie.isnull().sum())
    n_validos  = n_total - n_nulos
    n_categorias = serie.nunique(dropna=True)

    moda_series = serie.mode()
    moda_val    = moda_series.iloc[0] if not moda_series.empty else np.nan

    if pd.isna(moda_val):
        freq_moda   = 0
        pct_moda    = 0.0
        top3_str    = "Sem dados"
    else:
        freq_moda = int((serie == moda_val).sum())
        pct_moda  = round(freq_moda / n_validos * 100, 1) if n_validos > 0 else 0.0
        top3      = serie.value_counts().head(3)
        top3_str  = " | ".join([
            f"Cat '{k}': {v}x ({round(v/n_validos*100,1)}%)"
            for k, v in top3.items()
        ])

    registros_quali.append({
        'Variável':         col,
        'Tipo':             'Ordinal' if col in qual_ordinais else 'Nominal',
        'N Válidos':        n_validos,
        'Qtd Vazios':       n_nulos,
        '% Vazios':         round(n_nulos / n_total * 100, 1),
        'N Categorias':     n_categorias,
        'Moda':             moda_val,
        'Freq da Moda':     freq_moda,
        '% Moda':           pct_moda,
        'Top 3 Frequências': top3_str,
    })

tabela_quali = pd.DataFrame(registros_quali).set_index('Variável')

print(f"Variáveis qualitativas: {len(tabela_quali)}")
display(tabela_quali)

Variáveis qualitativas: 115


,Tipo,N Válidos,Qtd Vazios,% Vazios,N Categorias,Moda,Freq da Moda,% Moda,Top 3 Frequências
Variável,,,,,,,,,
N010,Ordinal,19873,0,0.0,4,1.0,15292,76.9,Cat '1.0': 15292x (76.9%) | Cat '2.0': 2719x (...
N011,Ordinal,19873,0,0.0,4,1.0,14533,73.1,Cat '1.0': 14533x (73.1%) | Cat '2.0': 3763x (...
N012,Ordinal,19873,0,0.0,4,1.0,16110,81.1,Cat '1.0': 16110x (81.1%) | Cat '2.0': 2871x (...
N013,Ordinal,19873,0,0.0,4,1.0,17256,86.8,Cat '1.0': 17256x (86.8%) | Cat '2.0': 1979x (...
N014,Ordinal,19873,0,0.0,4,1.0,17308,87.1,Cat '1.0': 17308x (87.1%) | Cat '2.0': 1687x (...
...,...,...,...,...,...,...,...,...,...
Estado,Nominal,19873,0,0.0,27,São Paulo,1244,6.3,Cat 'São Paulo': 1244x (6.3%) | Cat 'Minas Ger...
Regiao,Nominal,19873,0,0.0,5,Nordeste,6618,33.3,Cat 'Nordeste': 6618x (33.3%) | Cat 'Sudeste':...
E01201,Nominal,15660,4213,21.2,382,9111.0,955,6.1,Cat '9111.0': 955x (6.1%) | Cat '6111.0': 625x...


In [54]:
grupos = {
    'Sono':                  cols_sono,
    'Sintomas psicológicos': cols_sintomas_psicologicos,
    'Drogas':                cols_drogas,
    'Atividade física':      cols_atividade,
    'Dieta':                 cols_dieta,
    'Deficiências':          cols_deficiencia,
    'Gravidez':              cols_gravidez,
    'Comorbidades / TOC':    cols_comorbidades,
    'Violência':             cols_violencia,
    'Serviços de saúde':     cols_servicos_saude,
    'Trabalho / Renda':      cols_trabalho_renda_demografia,
    'Antropometria':         cols_antropometria,
}

resumo_grupos = []
for nome, lista in grupos.items():
    presentes = [c for c in lista if c in df_juncao.columns]
    if not presentes:
        continue
    sub = df_juncao[presentes]
    total_cells = sub.size
    total_nulos = sub.isnull().sum().sum()
    pct_nulos   = round(total_nulos / total_cells * 100, 1) if total_cells > 0 else 0
    col_mais_nula = sub.isnull().sum().idxmax()
    max_nulos_col = sub.isnull().sum().max()
    resumo_grupos.append({
        'Grupo':                nome,
        'N Variáveis':          len(presentes),
        'Total Células':        total_cells,
        'Total Nulos':          total_nulos,
        '% Nulos (grupo)':      pct_nulos,
        'Coluna Mais Nula':     col_mais_nula,
        'Nulos Coluna Pior':    max_nulos_col,
    })

tabela_nulos = pd.DataFrame(resumo_grupos).set_index('Grupo')
display(tabela_nulos)

,N Variáveis,Total Células,Total Nulos,% Nulos (grupo),Coluna Mais Nula,Nulos Coluna Pior
Grupo,,,,,,
Sono,1,19873,0,0.0,N010,0
Sintomas psicológicos,8,158984,0,0.0,N011,0
Drogas,8,158984,79467,50.0,P051,19628
Atividade física,8,158984,66972,42.1,P03904,12632
Dieta,40,794920,49659,6.2,P02101,12995
Deficiências,11,218603,20171,9.2,G046,12433
Gravidez,2,39746,20816,52.4,P005,10408
Comorbidades / TOC,18,357714,39335,11.0,Q11007,19664
Violência,8,158984,34196,21.5,V034,17098
